In [ ]:
# =========================
# CAMERA FOLDER SCANNER
# =========================

import os
from pathlib import Path
import pandas as pd

# ---- USER INPUT ----
base_path = Path(r"C:\YOUR\BASE\DIRECTORY")  # <-- CHANGE THIS
frame_rate = 60.0  # Hz

# file extensions to count (adjust if needed)
frame_extensions = [".png", ".jpg", ".jpeg", ".tif", ".bmp"]

# ---- FIND SESSION FOLDERS ----
session_folders = sorted([p for p in base_path.iterdir() if p.is_dir()])

results = []

for folder in session_folders:
    # look inside cam folders (cam1, cam2, etc.)
    cam_folders = [p for p in folder.iterdir() if p.is_dir()]
    
    for cam in cam_folders:
        # count frames
        frame_files = [
            f for f in cam.iterdir()
            if f.suffix.lower() in frame_extensions
        ]
        
        n_frames = len(frame_files)
        duration_sec = n_frames / frame_rate
        
        results.append({
            "session_folder": folder.name,
            "camera": cam.name,
            "n_frames": n_frames,
            "duration_sec": duration_sec,
            "duration_min": duration_sec / 60
        })

# ---- SAVE OUTPUT ----
df = pd.DataFrame(results)

output_path = base_path / "camera_summary.csv"
df.to_csv(output_path, index=False)

print(df)
print(f"\nSaved to: {output_path}")

In [1]:
# =========================
# TTL → CAMERA MATCHING TOOL
# =========================

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# =========================
# ---- USER INPUT ----
# =========================

# TTL folder (your path)
ttl_path = Path(r"C:\Users\Shermanlab\Desktop\2026-02-26_14-01-37\Record Node 104\experiment2\recording1\events\NI-DAQmx-103.PXIe-6341\TTL")

# camera summary CSV from previous script
camera_csv = Path(r"C:\YOUR\BASE\DIRECTORY\camera_summary.csv")  # <-- CHANGE

sample_rate = 30000  # Hz
expected_hz = 60.0

# tolerance for matching (seconds)
match_tolerance = 0.5

# gap threshold to separate blocks (seconds)
gap_threshold = 0.5


# =========================
# ---- LOAD TTL DATA ----
# =========================

# Open Ephys TTL format
timestamps = np.load(ttl_path / "timestamps.npy") / sample_rate
states = np.load(ttl_path / "states.npy")

# keep only rising edges
rising = states > 0
ttl_times = timestamps[rising]

print(f"Loaded {len(ttl_times)} TTL pulses")


# =========================
# ---- DETECT DROPPED FRAMES ----
# =========================

isi = np.diff(ttl_times)
expected_isi = 1 / expected_hz

dropped = isi > (expected_isi * 1.5)

print(f"Dropped frames detected: {np.sum(dropped)}")

# OPTIONAL: print where they occur
drop_times = ttl_times[:-1][dropped]


# =========================
# ---- FIND TTL BLOCKS ----
# =========================

gaps = np.diff(ttl_times)
block_edges = np.where(gaps > gap_threshold)[0]

blocks = []
start_idx = 0

for edge in block_edges:
    end_idx = edge
    blocks.append(ttl_times[start_idx:end_idx+1])
    start_idx = edge + 1

# last block
blocks.append(ttl_times[start_idx:])


# =========================
# ---- COMPUTE BLOCK STATS ----
# =========================

ttl_summary = []

for i, b in enumerate(blocks):
    duration = b[-1] - b[0]
    n_pulses = len(b)
    
    ttl_summary.append({
        "block_id": i,
        "n_pulses": n_pulses,
        "duration_sec": duration,
        "duration_min": duration / 60
    })

ttl_df = pd.DataFrame(ttl_summary)

print("\nTTL BLOCKS:")
print(ttl_df)


# =========================
# ---- LOAD CAMERA DATA ----
# =========================

cam_df = pd.read_csv(camera_csv)


# =========================
# 🔥 SOFT ORDER MATCHING (forward-only, skip allowed)
# =========================

cam_df_sorted = cam_df.sort_values("session_folder").reset_index(drop=True)

matches = []
cam_idx = 0  # only moves forward

for _, ttl_row in ttl_df.iterrows():
    ttl_dur = ttl_row["duration_sec"]
    ttl_pulses = ttl_row["n_pulses"]
    
    best_match = None
    best_score = np.inf
    best_idx = None
    
    # search forward only
    for i in range(cam_idx, len(cam_df_sorted)):
        cam_row = cam_df_sorted.iloc[i]
        
        cam_dur = cam_row["duration_sec"]
        cam_frames = cam_row["n_frames"]
        
        # --- combined score ---
        dur_diff = abs(ttl_dur - cam_dur)
        count_diff = abs(ttl_pulses - cam_frames)
        
        score = dur_diff + (count_diff / expected_hz)
        
        if score < best_score:
            best_score = score
            best_match = cam_row
            best_idx = i
    
    # accept only if reasonable
    if best_match is not None and best_score < match_tolerance:
        matches.append({
            "block_id": ttl_row["block_id"],
            "ttl_duration": ttl_dur,
            "camera_folder": best_match["session_folder"],
            "camera_duration": best_match["duration_sec"],
            "score": best_score
        })
        
        cam_idx = best_idx + 1  # 🔥 move forward
        
    else:
        matches.append({
            "block_id": ttl_row["block_id"],
            "ttl_duration": ttl_dur,
            "camera_folder": "NO MATCH",
            "camera_duration": np.nan,
            "score": np.nan
        })

match_df = pd.DataFrame(matches)

print("\nMATCHES:")
print(match_df)


# =========================
# ---- PLOT TTL + MATCHES ----
# =========================

plt.figure(figsize=(14, 4))

# plot TTL pulses as vertical ticks
plt.eventplot(ttl_times, lineoffsets=1, linelengths=0.8)

# annotate blocks
for i, b in enumerate(blocks):
    t0 = b[0]
    t1 = b[-1]
    
    match_row = match_df.loc[match_df["block_id"] == i].iloc[0]
    label = match_row["camera_folder"]
    
    plt.fill_between([t0, t1], 0, 0.5, alpha=0.3)
    
    plt.text(
        (t0 + t1)/2,
        0.2,
        label,
        ha="center",
        va="center",
        fontsize=9,
        rotation=0
    )

plt.ylim(0, 1.5)
plt.xlabel("Time (s)")
plt.title("TTL Pulses with Matched Camera Segments")
plt.yticks([])
plt.tight_layout()
plt.show()

Loaded 85614 TTL pulses
Dropped frames detected: 0

TTL BLOCKS:
   block_id  n_pulses  duration_sec  duration_min
0         0     85614      0.024233      0.000404


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\YOUR\\BASE\\DIRECTORY\\camera_summary.csv'